# 02 — Pipeline RAG e Exploração da Voz do Cliente (VoC)

Este notebook demonstra o funcionamento completo do pipeline **Retrieval-Augmented Generation (RAG)** desenvolvido para explorar as avaliações de clientes da base Olist:

1. **Transformação da pergunta (*Multi-Query Expansion*):** geração de variações semânticas para ampliar o *recall* sobre termos informais usados pelos consumidores.
2. **Recuperação semântica e seleção:** busca vetorial no `FAISS` com normalização L2/cosseno, deduplicação por `review_id` e filtro de relevância mínima (`RAG_RELEVANCE_THRESHOLD = 0.30`).
3. **Construção do contexto e Geração Fundamentada:** formatação das evidências com metadados e síntese via `ChatOpenAI` (`gpt-4o-mini`) sob *system prompt* estrito de *grounding*.
4. **Tratamento de perguntas sem evidências suficientes:** verificação em dupla camada (*Retrieval Gate* pré-LLM e *Guardrail* pós-LLM) para recusar perguntas fora do escopo sem alucinar.

## 1. Carregamento dos Exemplos Avaliados no Pipeline RAG

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EVALUATION_EXAMPLES_PATH

examples = json.loads(EVALUATION_EXAMPLES_PATH.read_text(encoding="utf-8"))
print(f"Total de cenários de exploração carregados: {len(examples)}")
for idx, item in enumerate(examples, start=1):
    tag = "FUNDAMENTADA" if item["has_sufficient_evidence"] else "RECUSA SEGURA"
    print(f"{idx}. [{tag}] {item['question']}")

Total de cenários de exploração carregados: 6
1. [FUNDAMENTADA] Quais são os principais problemas relatados pelos clientes em suas avaliações?
2. [FUNDAMENTADA] Quais reclamações relacionadas à entrega aparecem com maior frequência?
3. [FUNDAMENTADA] O que os clientes mais elogiam em suas experiências de compra?
4. [FUNDAMENTADA] Quais padrões podem ser identificados nas avaliações de clientes insatisfeitos?
5. [FUNDAMENTADA] Quais avaliações sustentam a conclusão de que existem problemas recorrentes relacionados ao prazo de entrega?
6. [RECUSA SEGURA] Qual é a cotação atual do dólar e a previsão do tempo em Tóquio amanhã?


## 2. Inspeção Detalhada: Transformação, Evidências Recuperadas e Resposta Fundamentada

In [2]:
entrega_case = examples[1]
print("=" * 80)
print(f"PERGUNTA: {entrega_case['question']}")
print("=" * 80)
print("\n[Consultas Expandidas (Multi-Query)]:")
for q in entrega_case["expanded_queries"]:
    print(f"  - {q}")

print(f"\n[Evidências Recuperadas ({entrega_case['retrieved_count']})]:")
for idx, ev in enumerate(entrega_case["evidences"][:3], start=1):
    print(f"  {idx}. review_id: {ev['review_id']} | Nota: {ev['review_score']}/5 | Relevância: {ev['relevance_score']:.3f}")
    print(f"     \"{ev['text']}\"")

print("\n[Resposta Gerada pelo LLM]:")
print(entrega_case["answer"])

PERGUNTA: Quais reclamações relacionadas à entrega aparecem com maior frequência?

[Consultas Expandidas (Multi-Query)]:
  - Quais reclamações relacionadas à entrega aparecem com maior frequência?
  - demora na entrega atraso correios produto não chegou no prazo
  - pedido consta como entregue mas não recebi extravio
  - entrega parcial faltou item do pedido

[Evidências Recuperadas (5)]:
  1. review_id: d7ea05bb4f9c77d606fd90b1a27e317d | Nota: 1/5 | Relevância: 0.864
     "Comprei o produto dia 25 de fevereiro e hoje dia 29 de marco não fora entregue na minha residência. Não sei se os correios desse Brasil e péssimo ou foi a própria loja que demorou postar."
  2. review_id: 185cb85ede91009d7a8e685687c037ee | Nota: 1/5 | Relevância: 0.851
     "fiz minha compra faz 30 dias e não recebi ainda meu produto. Precisa melhorar nas entregas"
  3. review_id: 06f47dacd0575e592bf6a17e922ce6cd | Nota: 1/5 | Relevância: 0.839
     "não recebi ainda. aqui está descrevendo como entregue só que ate a

## 3. Verificação do Tratamento de Perguntas Fora de Escopo / Sem Evidências

In [3]:
refusal_case = examples[-1]
print(f"Pergunta: {refusal_case['question']}")
print(f"Evidências suficientes? {refusal_case['has_sufficient_evidence']}")
print(f"Motivo do bloqueio:     {refusal_case['refusal_reason']}")
print(f"Documentos recuperados: {refusal_case['retrieved_count']}")
print(f"Resposta do sistema:    {refusal_case['answer']}")

Pergunta: Qual é a cotação atual do dólar e a previsão do tempo em Tóquio amanhã?
Evidências suficientes? False
Motivo do bloqueio:     below_relevance_threshold
Documentos recuperados: 0
Resposta do sistema:    Com base nas avaliações disponíveis na base de conhecimento da Olist, não foram encontradas evidências suficientes para responder a esta pergunta com segurança. A questão pode estar fora do escopo dos relatos de clientes ou não possuir ocorrências relevantes registradas.


### Q&A
- **Quais são os principais problemas e reclamações de entrega relatados pelos clientes?** Os consumidores apontam atrasos superiores a 30 dias (`review_id: 185cb85ede91009d7a8e685687c037ee`), pedidos registrados como entregues no rastreamento sem terem chegado ao destinatário (`review_id: 06f47dacd0575e592bf6a17e922ce6cd`), entregas parciais faltando itens (`review_id: ae728c1061bf163b4bd256ad9ee0bb83`), produtos avariados ou divergentes do anúncio (`review_id: 6f885e2fd69412264d9e74e805175d53`) e demora no reembolso após devolução (`review_id: 2d0d80576ee9bbab49cc108d9ec7a0fb`).
- **O que os clientes mais elogiam em suas experiências de compra?** A entrega realizada antes do prazo estimado (`review_id: fe3db7c069d694bab50cc43463f91608`), a excelente qualidade do produto adquirido (`review_id: 5241fcad10d7b45a1761f1e1e1536faa`) e a eficiência no atendimento do lojista (`review_id: 07b48dbc11677f5021dbefcaba550ed2`).
- **Como o sistema reage a perguntas sem evidências suficientes?** Quando a consulta não atinge o limiar mínimo de similaridade semântica (`0.30`) ou está fora do domínio da Olist, o pipeline interrompe a geração e retorna uma recusa transparente sem inventar informações.

### Data Analysis Key Findings
- A expansão *Multi-Query* permite recuperar avaliações relevantes com similaridade de cosseno entre `0.796` e `0.881` mesmo quando o usuário utiliza termos genéricos de negócio e o cliente escreveu relatos coloquiais.
- Todas as respostas válidas geradas pelo pipeline preservam rastreabilidade direta por meio do par `[review_id, Nota]`, permitindo auditar o comentário exato que originou cada conclusão executiva.

### Insights or Next Steps
- Os relatos de insatisfação severa (nota 1) combinam falhas logísticas com ausência de comunicação ativa no pós-venda, indicando que alertas automáticos em atrasos e divergências de status de entrega têm alto potencial de redução de *churn*.
- Novas perguntas ad-hoc podem ser executadas diretamente via CLI com `python -m src.cli query -q "<sua pergunta>"`.